# Pipeline SDC — Analyse de métadonnées par LLM

Ce notebook présente le pipeline `metadata-analysis-llm-for-sdc`.

Run pip install -r requirements-notebook.txt

In [6]:
# pour changer de modèle (autre que qwen dans ce cas)
import os 
os.environ["LLM_MODEL"] = "gemma4-26b-moe"

## Prérequis. Mise en place

> **Prérequis Onyxia :** service VSCode-python lancé avec `CLE_API_OPENWEBUI` déclarée en variable d'environnement, et `pip install -r requirements.txt` exécuté dans le terminal.

In [7]:
import os, sys, tempfile
from pathlib import Path
import s3fs
from IPython.display import Markdown, display

# Détection de la racine du repo (fonctionne que le notebook soit lancé
# depuis la racine ou depuis notebooks/)
for _p in [Path.cwd(), Path.cwd().parent]:
    if (_p / "core").is_dir():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Impossible de localiser 'core/'. Lancez Jupyter depuis la racine du repo.")

sys.path.insert(0, str(REPO_ROOT))
from core import pipeline, transform_output

# Vérification de l'environnement
api_key = os.environ.get("CLE_API_OPENWEBUI") or os.environ.get("OPENAI_API_KEY")
assert api_key, "Clé API introuvable — déclarez CLE_API_OPENWEBUI dans les variables d'environnement Onyxia."
print(f"✓ Clé API       : {'*' * (len(api_key))}")
print(f"✓ Modèle        : {os.environ.get('LLM_MODEL', 'qwen3-6-35b-moe')}")

fs = s3fs.S3FileSystem(
    client_kwargs={"endpoint_url": "https://" + os.environ["AWS_S3_ENDPOINT"]}
)
print("✓ MinIO         : connexion initialisée")

✓ Clé API       : ***********************************
✓ Modèle        : gemma4-26b-moe
✓ MinIO         : connexion initialisée


## 1. Sérialisation — Lecture et transformation markdown

`transform_input.serialize` convertit le classeur en Markdown pur.


In [8]:
INPUT_S3  = "s3://jawadmallat/analyse_LLM_metadata/data_tables/metadonnees_ex5_sansSL2.ods"
OUTPUT_S3 = "s3://jawadmallat/analyse_LLM_metadata/output/metadonnees_ex5_sansSL2.csv" # le chemin du fichier de sortie, qui est attendu en csv

# Téléchargement depuis MinIO vers /tmp
tmp_input = Path(f"/tmp/input{Path(INPUT_S3).suffix}")
with fs.open(INPUT_S3, "rb") as f_in:
    tmp_input.write_bytes(f_in.read())
print(f"✓ Téléchargé : {INPUT_S3}\n")

# Sérialisation
md_input = pipeline.serialize(tmp_input)
print(f"  {len(md_input)} caractères — aperçu :")
display(Markdown(md_input))

✓ Téléchargé : s3://jawadmallat/analyse_LLM_metadata/data_tables/metadonnees_ex5_sansSL2.ods

  28370 caractères — aperçu :


# input.ods


## Tableaux

| Nom du tableau | Libellé | Note Haut | Note Bas | Variables en ligne | Variables en colonne | Variables en page | Décimales |
| --- | --- | --- | --- | --- | --- | --- | --- |
| naf_T1 | Consommation d'énergie en milliers de tonnes-équivalent-pétrole (kTEP) et nombre d'établissements selon le secteur d'activité en NAF. rév.2 |  | Champ : France, établissements de 20 salariés ou plus appartenant à l'industrie, hors industrie de l'énergie et hors artisanat commercial, y compris récupération. Source : Insee, Enquête annuelle sur les consommations d'énergie dans l'industrie 2021. | SECTEUR | INDIC_T1 |  | 1 |
| naf_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyens des produits énergétiques selon le secteur d'activité en NAF rév.2 de l'établissement |  | Note : la consommation d'énergie peut différer de la quantité achetée dans le cas d'énergies stockables (fioul par exemple), ou dans le cas de l'électricité (autoproduction consommée) et du bois (une partie non achetée). Champ : France, établissements de 20 salariés ou plus appartenant à l'industrie, hors industrie de l'énergie et hors artisanat commercial, y compris récupération. Source : Insee, Enquête annuelle sur les consommations d'énergie dans l'industrie 2021. | SECTEUR | INDIC_T2 |  | 1 |
| naf_T3 | Répartition de la consommation de combustibles par usage en milliers de tonnes-équivalent-pétrole (kTEP) selon le secteur d'activité en NAF rév.2 de l'établissement |  | Champ : France, établissements de 20 salariés ou plus appartenant à l'industrie, hors industrie de l'énergie et hors artisanat commercial, y compris récupération. Source : Insee, Enquête annuelle sur les consommations d'énergie dans l'industrie 2021. | SECTEUR | INDIC_T3 |  | 1 |
| naf_T4 | Autoproduction, achats et consommation d'électricité par usage en GWh selon le secteur d'activité en NAF rév.2 de l'établissement |  | Champ : France, établissements de 20 salariés ou plus appartenant à l'industrie, hors industrie de l'énergie et hors artisanat commercial, y compris récupération. Source : Insee, Enquête annuelle sur les consommations d'énergie dans l'industrie 2021. | SECTEUR | INDIC_T4 |  | 1 |
| reg_T1 | Consommation d'énergie en milliers de tonnes-équivalent-pétrole (kTEP) et nombre d'établissements selon la région |  | Champ : France, établissements de 20 salariés ou plus appartenant à l'industrie, hors industrie de l'énergie et hors artisanat commercial, y compris récupération. Source : Insee, Enquête annuelle sur les consommations d'énergie dans l'industrie 2021. | REGION | INDIC_T1 |  | 1 |
| reg_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyens des produits énergétiques selon la région |  | Note : la consommation d'énergie peut différer de la quantité achetée dans le cas d'énergies stockables (fioul par exemple), ou dans le cas de l'électricité (autoproduction consommée) et du bois (une partie non achetée). Champ : France, établissements de 20 salariés ou plus appartenant à l'industrie, hors industrie de l'énergie et hors artisanat commercial, y compris récupération. Source : Insee, Enquête annuelle sur les consommations d'énergie dans l'industrie 2021. | REGION | INDIC_T2 |  | 1 |
| reg_T3 | Répartition de la consommation de combustibles par usage en milliers de tonnes-équivalent-pétrole (kTEP) selon la région |  | Champ : France, établissements de 20 salariés ou plus appartenant à l'industrie, hors industrie de l'énergie et hors artisanat commercial, y compris récupération. Source : Insee, Enquête annuelle sur les consommations d'énergie dans l'industrie 2021. | REGION | INDIC_T3 |  | 1 |
| reg_T4 | Autoproduction, achats et consommation d'électricité par usage en GWh selon la région |  | Champ : France, établissements de 20 salariés ou plus appartenant à l'industrie, hors industrie de l'énergie et hors artisanat commercial, y compris récupération. Source : Insee, Enquête annuelle sur les consommations d'énergie dans l'industrie 2021. | REGION | INDIC_T4 |  | 1 |
| teff_T1 | Consommation d'énergie en milliers de tonnes-équivalent-pétrole (kTEP) et nombre d'établissements selon la tranche d'effectif de l'établissement |  | Champ : France, établissements de 20 salariés ou plus appartenant à l'industrie, hors industrie de l'énergie et hors artisanat commercial, y compris récupération. Source : Insee, Enquête annuelle sur les consommations d'énergie dans l'industrie 2021. | TEFF | INDIC_T1 |  | 1 |
| teff_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyens des produits énergétiques selon la tranche d'effectif de l'établissement |  | Note : la consommation d'énergie peut différer de la quantité achetée dans le cas d'énergies stockables (fioul par exemple), ou dans le cas de l'électricité (autoproduction consommée) et du bois (une partie non achetée). Champ : France, établissements de 20 salariés ou plus appartenant à l'industrie, hors industrie de l'énergie et hors artisanat commercial, y compris récupération. Source : Insee, Enquête annuelle sur les consommations d'énergie dans l'industrie 2021. | TEFF | INDIC_T2 |  | 1 |
| teff_T3 | Répartition de la consommation de combustibles par usage en milliers de tonnes-équivalent-pétrole (kTEP) selon la tranche d'effectif de l'établissement |  | Champ : France, établissements de 20 salariés ou plus appartenant à l'industrie, hors industrie de l'énergie et hors artisanat commercial, y compris récupération. Source : Insee, Enquête annuelle sur les consommations d'énergie dans l'industrie 2021. | TEFF | INDIC_T3 |  | 1 |
| teff_T4 | Autoproduction, achats et consommation d'électricité par usage en GWh selon la tranche d'effectif de l'établissement |  | Champ : France, établissements de 20 salariés ou plus appartenant à l'industrie, hors industrie de l'énergie et hors artisanat commercial, y compris récupération. Source : Insee, Enquête annuelle sur les consommations d'énergie dans l'industrie 2021. | TEFF | INDIC_T4 |  | 1 |

## Variables

| Variable | Description | Modalité | Définition | Regroupement | Libellé 1 |
| --- | --- | --- | --- | --- | --- |
| REGION | Région | global |  | 11 24 27 28 32 44 52 53 75 76 84 93 97 | Toutes régions |
| REGION | Région | 11 |  |  | Île-de-France |
| REGION | Région | 24 |  |  | Centre-Val de Loire |
| REGION | Région | 27 |  |  | Bourgogne-Franche-Comté |
| REGION | Région | 28 |  |  | Normandie |
| REGION | Région | 32 |  |  | Hauts-de-France |
| REGION | Région | 44 |  |  | Grand Est |
| REGION | Région | 52 |  |  | Pays de la Loire |
| REGION | Région | 53 |  |  | Bretagne |
| REGION | Région | 75 |  |  | Nouvelle-Aquitaine |
| REGION | Région | 76 |  |  | Occitanie |
| REGION | Région | 84 |  |  | Auvergne-Rhône-Alpes |
| REGION | Région | 93 |  |  | Provence-Alpes-Côte d'Azur et Corse |
| REGION | Région | 97 |  |  | Dom |
| SECTEUR | Secteur d'activité, division de la NAF rév.2 | global |  | 07 08 09 10 11 12 13 14 15 16 17 18 20 21 22 23 24 25 26 27 28 29 30 31 32 33 38 | Total industrie |
| SECTEUR | Secteur d'activité, division de la NAF rév.2 | 07-09 |  |  | 07 - 08 – 09 Toutes les Industries extractives |
| SECTEUR | Secteur d'activité, division de la NAF rév.2 | 10-12 |  |  | 10 – 11 – 12 Industries alimentaires, Fabrication de boissons, Fabrication de produits à base de tabac |
| SECTEUR | Secteur d'activité, division de la NAF rév.2 | 13 |  |  | 13 - Fabrication de textiles |
| SECTEUR | Secteur d'activité, division de la NAF rév.2 | 14 |  |  | 14 - Industrie de l'habillement |
| SECTEUR | Secteur d'activité, division de la NAF rév.2 | 15 |  |  | 15 - Industrie du cuir et de la chaussure |
| SECTEUR | Secteur d'activité, division de la NAF rév.2 | 16 |  |  | 16 - Travail du bois et fabrication d'articles en bois et en liège, à l'exception des meubles - fabrication d'articles en vannerie et sparterie |
| SECTEUR | Secteur d'activité, division de la NAF rév.2 | 17 |  |  | 17 - Industrie du papier et du carton |
| SECTEUR | Secteur d'activité, division de la NAF rév.2 | 18 |  |  | 18 - Imprimerie et reproduction d'enregistrements |
| SECTEUR | Secteur d'activité, division de la NAF rév.2 | 20 |  |  | 20 - Industrie chimique |
| SECTEUR | Secteur d'activité, division de la NAF rév.2 | 21 |  |  | 21 - Industrie pharmaceutique |
| SECTEUR | Secteur d'activité, division de la NAF rév.2 | 22 |  |  | 22 - Fabrication de produits en caoutchouc et en plastique |
| SECTEUR | Secteur d'activité, division de la NAF rév.2 | 23 |  |  | 23 - Fabrication d'autres produits minéraux non métalliques |
| SECTEUR | Secteur d'activité, division de la NAF rév.2 | 24 |  |  | 24 - Métallurgie |
| SECTEUR | Secteur d'activité, division de la NAF rév.2 | 25 |  |  | 25 - Fabrication de produits métalliques, à l'exception des machines et des équipements |
| SECTEUR | Secteur d'activité, division de la NAF rév.2 | 26 |  |  | 26 - Fabrication de produits informatiques, électroniques et optiques |
| SECTEUR | Secteur d'activité, division de la NAF rév.2 | 27 |  |  | 27 - Fabrication d'équipements électriques |
| SECTEUR | Secteur d'activité, division de la NAF rév.2 | 28 |  |  | 28 - Fabrication de machines et équipements n.c.a. |
| SECTEUR | Secteur d'activité, division de la NAF rév.2 | 29 |  |  | 29 - Industrie automobile |
| SECTEUR | Secteur d'activité, division de la NAF rév.2 | 30 |  |  | 30 - Fabrication d'autres matériels de transport |
| SECTEUR | Secteur d'activité, division de la NAF rév.2 | 31 |  |  | 31 - Fabrication de meubles |
| SECTEUR | Secteur d'activité, division de la NAF rév.2 | 32 |  |  | 32 - Autres industries manufacturières |
| SECTEUR | Secteur d'activité, division de la NAF rév.2 | 33 |  |  | 33 - Réparation et installation de machines et d'équipements |
| SECTEUR | Secteur d'activité, division de la NAF rév.2 | 38 |  |  | 38 - Récupération |
| TEFF | Tranche d'effectif salarié | global | Intitulé | 03 04 05 06 07 | Total industrie |
| TEFF | Tranche d'effectif salarié | 03 |  |  | 20 à 49 salariés |
| TEFF | Tranche d'effectif salarié | 04 |  |  | 50 à 99 salariés |
| TEFF | Tranche d'effectif salarié | 05 |  |  | 100 à 249 salariés |
| TEFF | Tranche d'effectif salarié | 06 |  |  | 250 à 499 salariés |
| TEFF | Tranche d'effectif salarié | 07 |  |  | 500 salariés ou plus |

## Indicateurs

| Variable | Libellé | Modalité | Notes des modalités | Décimales | Libellé 1 |
| --- | --- | --- | --- | --- | --- |
| INDIC_T1 | Consommation d'énergie en ktep | NB_ETAB |  | 0 | Nombre d'établissements |
| INDIC_T1 | Consommation d'énergie en ktep | CMSR_CSTP | Total des combustibles minéraux solides = houille + lignite charbon pauvre + coke de houille | 1 | Combustibles minéraux solides (houille, lignite - charbon pauvre, coke de houille) |
| INDIC_T1 | Consommation d'énergie en ktep | GAZR_CSTP |  | 1 | Gaz naturel et autres gaz |
| INDIC_T1 | Consommation d'énergie en ktep | BIOR_CSTP |  | 1 | Biogaz et biométhane |
| INDIC_T1 | Consommation d'énergie en ktep | IR_CSTP |  | 1 | Coke de pétrole |
| INDIC_T1 | Consommation d'énergie en ktep | JR_CSTP |  | 1 | Butane propane |
| INDIC_T1 | Consommation d'énergie en ktep | KR_CSTP |  | 1 | Fioul lourd |
| INDIC_T1 | Consommation d'énergie en ktep | LR_CSTP |  | 1 | Fioul domestique |
| INDIC_T1 | Consommation d'énergie en ktep | QR_CSTP |  | 1 | Gazole non routier |
| INDIC_T1 | Consommation d'énergie en ktep | CUR_CSTP | Total combustibles usuels = Combustibles minéraux solides + gaz de réseau + coke de pétrole + butane propane + fioul lourd + fioul domestique + gazole non routier. | 1 | Total combustibles usuels |
| INDIC_T1 | Consommation d'énergie en ktep | MR_CSTP |  | 1 | Autres produits pétroliers |
| INDIC_T1 | Consommation d'énergie en ktep | NR_CSTP |  | 1 | Liqueurs noires |
| INDIC_T1 | Consommation d'énergie en ktep | OR_CSTP |  | 1 | Bois et sous-produit du bois |
| INDIC_T1 | Consommation d'énergie en ktep | PR_CSTP |  | 1 | Hydrogene |
| INDIC_T1 | Consommation d'énergie en ktep | XR_CSTP |  | 1 | Combustibles spéciaux renouvelables |
| INDIC_T1 | Consommation d'énergie en ktep | YR_CSTP |  | 1 | Combustibles spéciaux non renouvelables |
| INDIC_T1 | Consommation d'énergie en ktep | CAR_CSTP | Total autres combustibles = autres produits pétroliers + liqueur noire + bois et sous-produits du bois + hydrogène + combustibles spéciaux renouvelables + combustibles spéciaux non renouvelables. | 1 | Total autres combustibles |
| INDIC_T1 | Consommation d'énergie en ktep | CR_CSTP |  | 1 | Achats de vapeur |
| INDIC_T1 | Consommation d'énergie en ktep | BR_CSTP |  | 1 | Electricité |
| INDIC_T1 | Consommation d'énergie en ktep | CONSBRT | Total brut = total combustibles + achats de vapeur + électricité | 1 | Total brut |
| INDIC_T1 | Consommation d'énergie en ktep | CR_VTTP |  | 1 | Vapeur vendue |
| INDIC_T1 | Consommation d'énergie en ktep | COMBPRB |  | 1 | Consommation pour production d'électricité |
| INDIC_T1 | Consommation d'énergie en ktep | CONSNET | Total net = Total brut - Vapeur vendue - Consommation pour production d'électricité. | 1 | Total net |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | FACTURE | en millions d’euros hors taxes | 1 | Valeur des achats de l’ensemble des énergies (en millions d'euros) |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | CMSR_QAUP | en milliers de tonnes | 1 | Quantités achetées de combustibles minéraux solides |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | CMSR_CSUP | en milliers de tonnes | 1 | Quantités consommées de combustibles minéraux solides |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | CMSR_VALA | en millions d’euros, hors taxes | 1 | Valeur des achats de combustibles minéraux solides |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | CMSR_PRIX | en euros par tonne, hors taxes | 1 | Prix moyen des combustibles minéraux solides |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | GAZR_QAUP | en GWh | 1 | Quantités achetées de gaz |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | GAZR_CSUP | en GWh | 1 | Quantités consommées de gaz |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | GAZR_VALA | en millions d’euros hors taxes | 1 | Valeur des achats de gaz |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | GAZR_PRIX | en euros par MWh, hors taxes | 1 | Prix moyen du gaz |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | BIOR_QAUP | en GWh | 1 | Quantités achetées de biogaz et biométhane |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | BIOR_CSUP | en GWh | 1 | Quantités consommées de biogaz et biométhane |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | BIOR_VALA | en millions d’euros hors taxes | 1 | Valeur des achats de biogaz et biométhane |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | BIOR_PRIX | en euros par MWh, hors taxes | 1 | Prix moyen du biogaz et biométhane |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | IR_QAUP | en milliers de tonnes | 1 | Quantités achetées de coke de pétrole |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | IR_CSUP | en milliers de tonnes | 1 | Quantités consommées de coke de pétrole |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | IR_VALA | en millions d’euros, hors taxes | 1 | Valeur des achats de coke de pétrole |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | IR_PRIX | en euros par tonne, hors taxes | 1 | Prix moyen de coke de pétrole |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | JR_QAUP | en milliers de tonnes | 1 | Quantités achetées de butane-propane |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | JR_CSUP | en milliers de tonnes | 1 | Quantités consommées de butane-propane |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | JR_VALA | en millions d’euros hors taxes | 1 | Valeur des achats de butane-propane |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | JR_PRIX | en euros par tonne, hors taxes | 1 | Prix moyen de butane-propane |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | KR_QAUP | en milliers de tonnes | 1 | Quantités achetées de fioul lourd |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | KR_CSUP | en milliers de tonnes | 1 | Quantités consommées de fioul lourd |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | KR_VALA | en millions d’euros, hors taxes | 1 | Valeur des achats de fioul lourd |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | KR_PRIX | en euros par tonne, hors taxes | 1 | Prix moyen de fioul lourd |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | LR_QAUP | en milliers de litres | 1 | Quantités achetées de fioul domestique |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | LR_CSUP | en milliers de litres | 1 | Quantités consommées de fioul domestique |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | LR_VALA | en millions d’euros hors taxes | 1 | Valeur des achats de fioul domestique |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | LR_PRIX | en euros par millier de litres, hors taxes | 1 | Prix moyen de fioul domestique |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | QR_QAUP | en milliers de litres | 1 | Quantités achetées de gazole non routier |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | QR_CSUP | en milliers de litres | 1 | Quantités consommées de gazole non routier |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | QR_VALA | en millions d’euros hors taxes | 1 | Valeur des achats de gazole non routier |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | QR_PRIX | en euros par millier de litres, hors taxes | 1 | Prix moyen de gazole non routier |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | OR_QAUP | en milliers de tonnes | 1 | Quantités achetées de bois |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | OR_CSUP | en milliers de tonnes | 1 | Quantités consommées de bois |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | OR_VALA | en millions d’euros, hors taxes | 1 | Valeur des achats de bois |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | OR_PRIX | en euros par tonne, hors taxes | 1 | Prix moyen du bois |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | CR_QAUP | en milliers de tonnes | 1 | Quantités achetées de vapeur |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | CR_VALA | en millions d’euros, hors taxes | 1 | Valeur des achats de vapeur |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | CR_PRIX | en euros par tonne, hors taxes | 1 | Prix moyen de vapeur |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | PR_QAUP | en milliers de tonnes | 1 | Quantités achetées d’hydrogène |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | PR_CSUP | en milliers de tonnes | 1 | Quantités consommées d’hydrogène |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | PR_PROD | en milliers de tonnes | 1 | Quantités autoproduites d’hydrogène |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | PR_VALA | en millions d’euros hors taxes | 1 | Valeur des achats d’hydrogène |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | PR_PRIX | en euros par tonnes, hors taxe | 1 | Prix moyen d’hydrogène |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | PR_AUCO | en milliers de tonnes | 1 | Autoproduction consommée |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | PR_VEND | en milliers de tonnes | 1 | Autoproduction vendue d’hydrogène |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | BR_QAUP | en GWh | 1 | Quantités achetées d'électricité |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | BR_CSUP | en GWh | 1 | Quantités consommées d'électricité |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | BR_PROD | en GWh | 1 | Quantités autoproduites d’électricité |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | BR_VALA | en millions d’euros hors taxes | 1 | Valeur des achats d'électricité |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | BR_PRIX | en euros par MWh, hors taxes | 1 | Prix moyen d’électricité |
| INDIC_T3 | Répartition de la consommation de combustibles par usage en ktep | GAZR_USQFAB |  | 1 | Quantités de gaz consommées pour la fabrication |
| INDIC_T3 | Répartition de la consommation de combustibles par usage en ktep | GAZR_USQ2 |  | 1 | Quantités de gaz consommées en tant que matières premières |
| INDIC_T3 | Répartition de la consommation de combustibles par usage en ktep | GAZR_USQ3 |  | 1 | Quantités de gaz consommées pour la production d’électricité |
| INDIC_T3 | Répartition de la consommation de combustibles par usage en ktep | GAZR_USQ4 |  | 1 | Quantités de gaz consommées pour le chauffage ou un autre usage |
| INDIC_T3 | Répartition de la consommation de combustibles par usage en ktep | GAZR_USQ9 |  | 1 | Quantités de gaz consommées pour la mobilité sur site |
| INDIC_T3 | Répartition de la consommation de combustibles par usage en ktep | GAZR_CSTP |  | 1 | Quantités de gaz consommées |
| INDIC_T3 | Répartition de la consommation de combustibles par usage en ktep | LR_USQFAB |  | 1 | Quantités de fioul domestique consommées pour la fabrication |
| INDIC_T3 | Répartition de la consommation de combustibles par usage en ktep | LR_USQ2 |  | 1 | Quantités de fioul domestique consommées en tant que matières premières |
| INDIC_T3 | Répartition de la consommation de combustibles par usage en ktep | LR_USQ3 |  | 1 | Quantités de fioul domestique consommées pour la production d’électricité |
| INDIC_T3 | Répartition de la consommation de combustibles par usage en ktep | LR_USQ4 |  | 1 | Quantités de fioul domestique consommées pour le chauffage ou un autre usage |
| INDIC_T3 | Répartition de la consommation de combustibles par usage en ktep | LR_CSTP |  | 1 | Quantités de fioul domestique consommées |
| INDIC_T3 | Répartition de la consommation de combustibles par usage en ktep | QR_USQFAB |  | 1 | Quantités de gazole non routier consommées pour la fabrication |
| INDIC_T3 | Répartition de la consommation de combustibles par usage en ktep | QR_USQ2 |  | 1 | Quantités de gazole non routier consommées en tant que matières premières |
| INDIC_T3 | Répartition de la consommation de combustibles par usage en ktep | QR_USQ3 |  | 1 | Quantités de gazole non routier consommées pour la production d’électricité |
| INDIC_T3 | Répartition de la consommation de combustibles par usage en ktep | QR_USQ4 |  | 1 | Quantités de gazole non routier consommées pour le chauffage ou un autre usage |
| INDIC_T3 | Répartition de la consommation de combustibles par usage en ktep | QR_USQ9 |  | 1 | Quantités de gazole non routier consommées pour la mobilité sur site |
| INDIC_T3 | Répartition de la consommation de combustibles par usage en ktep | QR_CSTP |  | 1 | Quantités de gazole non routier consommées |
| INDIC_T3 | Répartition de la consommation de combustibles par usage en ktep | PR_USQ2 |  | 1 | Quantités d’hydrogène consommées en tant que matières premières |
| INDIC_T3 | Répartition de la consommation de combustibles par usage en ktep | PR_USQ8 |  | 1 | Quantités d’hydrogène utilisées pour la combustion (hors production d’électricité) |
| INDIC_T3 | Répartition de la consommation de combustibles par usage en ktep | PR_USQ3 |  | 1 | Quantités d’hydrogène utilisées pour la production d’électricité |
| INDIC_T3 | Répartition de la consommation de combustibles par usage en ktep | PR_USQ9 |  | 1 | Quantités d’hydrogène utilisées pour la mobilité sur site |
| INDIC_T3 | Répartition de la consommation de combustibles par usage en ktep | PR_CSTP |  | 1 | Quantités d’hydrogène consommées |
| INDIC_T4 | Autoproduction, achats et consommation d'électricité par usage en GWh | BR_THER |  | 1 | Autoproduction d'origine thermique |
| INDIC_T4 | Autoproduction, achats et consommation d'électricité par usage en GWh | BR_NTHE | Autoproduction d'origine non thermique = Autoproduction d'origine hydraulique, photovoltaique ou éolienne. | 1 | Autoproduction d'origine non thermique |
| INDIC_T4 | Autoproduction, achats et consommation d'électricité par usage en GWh | BR_PROD |  | 1 | Autoproduction électricité |
| INDIC_T4 | Autoproduction, achats et consommation d'électricité par usage en GWh | BR_VEND |  | 1 | Autoproduction vendue |
| INDIC_T4 | Autoproduction, achats et consommation d'électricité par usage en GWh | BR_AUCO |  | 1 | Autoproduction consommée (1) |
| INDIC_T4 | Autoproduction, achats et consommation d'électricité par usage en GWh | BR_QAUP |  | 1 | Achats (2) |
| INDIC_T4 | Autoproduction, achats et consommation d'électricité par usage en GWh | BR_CSUP |  | 1 | Consommation (1 + 2) |
| INDIC_T4 | Autoproduction, achats et consommation d'électricité par usage en GWh | BR_USQ1 |  | 1 | Usage force motrice |
| INDIC_T4 | Autoproduction, achats et consommation d'électricité par usage en GWh | BR_USQ2 |  | 1 | Usages thermiques (y compris production de froid) |
| INDIC_T4 | Autoproduction, achats et consommation d'électricité par usage en GWh | BR_USQ4 |  | 1 | Usage électrolyse |
| INDIC_T4 | Autoproduction, achats et consommation d'électricité par usage en GWh | BR_USQ6 |  | 1 | Usage chauffage des locaux et autres |
| INDIC_T4 | Autoproduction, achats et consommation d'électricité par usage en GWh | BR_USQ7 |  | 1 | Usages thermodynamiques hors chauffage des locaux (Prod. froid, PAC, CMV) |
| INDIC_T4 | Autoproduction, achats et consommation d'électricité par usage en GWh | BR_USQ8 |  | 1 | Usage éclairage |
| INDIC_T4 | Autoproduction, achats et consommation d'électricité par usage en GWh | BR_USQ9 |  | 1 | Usage mobilité sur site |


## 2. Phase 1 — le modèle analyse et pose ses questions

`pipeline.start()` envoie le Markdown au modèle avec les instructions.

Le modèle produit une liste de questions

In [9]:
print("Phase 1 — envoi au modèle...\n")
r = pipeline.start(md_input)

if r.auto_continued:
    print("Le modèle n'a posé aucune question — JSON produit directement.")
    print("Passez à la cellule Phase 2 (la cellule 'Réponses' sera ignorée).")
else:
    print("─" * 70)
    print(r.questions)
    print("─" * 70)
    print("\n→ Remplissez la variable `answers` dans la cellule ci-dessous, puis exécutez-la.")

Phase 1 — envoi au modèle...

──────────────────────────────────────────────────────────────────────
**Questions**

1. **Champ et population**
   1. Pour l'ensemble des tableaux, le champ est-il exactement : `france_etablissements_20_salaries_ou_plus_appartenant_a_l_industrie_hors_industrie_de_l_energie_et_hors_artisanat_commercial_y_compris_recuperation` ?

2. **Indicateurs et hiérarchies**
   2. Pour les indicateurs, confirmez-vous que les relations de somme décrites dans les notes (ex: `CUR_CSTP`, `CAR_CSTP`, `CONSBRT`, `CONSNET`, `BR_CSUP`, `CONSBRT`) doivent être transformées en hiérarchies avec des codes de type `hrc_combustibles_usuels`, `hrc_autres_combustibles`, `hrc_total_brut`, etc. ?

3. **Variables de croisement et nomenclatures**
   3. La variable `REGION` doit-elle être traitée comme une nomenclature de type `nuts_code` / `hrc_nuts` malgré l'absence du terme "NUTS" dans le fichier ?
──────────────────────────────────────────────────────────────────────

→ Remplissez la v

## 3a. Réponses du producteur

Remplissez la variable `answers` avec vos réponses aux questions ci-dessus
(une réponse par question, dans l'ordre), puis exécutez la cellule.


In [10]:
# Remplissez vos réponses ici avant d'exécuter cette cellule.
answers = """
1) le champ est 2023 
2) Non. hrc_indicator = NA. INDIC_T1 à INDIC_T4 sont des groupes d'indicateurs. Dans la fiche "Indicateurs", "modalité" représente un indicateur. 
3) non tu peux garder "region" avec "hrc_region"
"""

print("Réponses enregistrées — exécutez la cellule Phase 2.")

Réponses enregistrées — exécutez la cellule Phase 2.


## 3b. — JSON, validation et CSV final

`pipeline.answer()` envoie les réponses au modèle qui produit le JSON.
Le JSON est ensuite **validé contre le schéma**, puis
converti en tableau plat et écrit sur MinIO.


In [11]:
import pandas as pd

if r.auto_continued:
    # Phase 1 a déjà produit le JSON — pas besoin d'un second appel
    records = r.records
    print("Résultats issus de la Phase 1 (aucune question posée).\n")
else:
    print("Phase 2 — envoi des réponses au modèle...\n")
    records = pipeline.answer(r.history, answers)
    print(f"✓ {len(records)} enregistrement(s) validés contre le schéma.\n")

# Écriture du CSV sur MinIO
tmp_csv = Path("/tmp/output.csv")
cols, rows = transform_output.write_csv(records, tmp_csv)

with open(tmp_csv, "r", encoding="utf-8-sig") as f:
    csv_content = f.read()
with fs.open(OUTPUT_S3, "w", encoding="utf-8") as f_out:
    f_out.write(csv_content)

print(f"✓ CSV écrit sur MinIO : {OUTPUT_S3}")
print(f"  {len(rows)} lignes × {len(cols)} colonnes\n")

# Affichage du résultat
df = pd.read_csv(tmp_csv, encoding="utf-8-sig", keep_default_na=False)
display(df)

Phase 2 — envoi des réponses au modèle...

✓ 551 enregistrement(s) validés contre le schéma.

✓ CSV écrit sur MinIO : s3://jawadmallat/analyse_LLM_metadata/output/metadonnees_ex5_sansSL2.csv
  551 lignes × 7 colonnes



,table_name,field,hrc_field,indicator,hrc_indicator,spanning_1,hrc_spanning_1
0,naf_T1,2023,NA,NB_ETAB,NA,naf_code,hrc_naf
1,naf_T1,2023,NA,CMSR_CSTP,NA,naf_code,hrc_naf
2,naf_T1,2023,NA,GAZR_CSTP,NA,naf_code,hrc_naf
3,naf_T1,2023,NA,BIOR_CSTP,NA,naf_code,hrc_naf
4,naf_T1,2023,NA,IR_CSTP,NA,naf_code,hrc_naf
...,...,...,...,...,...,...,...
546,teff_T4,2023,NA,BR_USQ4,NA,teff_code,hrc_teff
547,teff_T4,2023,NA,BR_USQ6,NA,teff_code,hrc_teff
548,teff_T4,2023,NA,BR_USQ7,NA,teff_code,hrc_teff
549,teff_T4,2023,NA,BR_USQ8,NA,teff_code,hrc_teff
